# MiniMax-H3 A100 极速一键部署脚本 (SageAttention + ComfyUI + Cloudflare Tunnel)
> **运行说明**：
> 1. 确保已在 `Runtime` -> `Change runtime type` 中选择 **GPU: A100**。
> 2. 点击 `Runtime` -> `Run all` (快捷键 `Ctrl + F9`)。
> 3. 运行约 3~5 分钟后，脚本末尾会打印出专属的 **Cloudflare 公网隧道链接**（如 `https://xxxx.trycloudflare.com`）。
> 4. 将该链接复制粘贴回 Antigravity 客户端，即可全自动下发任务与下载视频回本地！

In [ ]:
# 1. 验证 GPU 显卡型号（确保分配到 A100 80GB SXM4）
!nvidia-smi

In [ ]:
# 2. 挂载 Google 云端硬盘（秒级读取预存模型，避免重复耗时下载）
from google.colab import drive
import os

drive.mount('/content/drive')

In [ ]:
# 3. 安装 ComfyUI 底座与核心高性能依赖
%cd /content
!git clone https://github.com/comfyanonymous/ComfyUI.git
%cd /content/ComfyUI
!pip install -q torch torchvision torchaudio --extra-index-url https://download.pytorch.org/whl/cu124
!pip install -q -r requirements.txt

# 4. 安装加速组件：SageAttention 与 EasyCache（使单条渲染耗时从 8 分钟大幅降至 3 分钟以内）
!pip install -q sageattention
!pip install -q diffusers accelerate sentencepiece protobuf torchvision

In [ ]:
# 5. 克隆 MiniMax-H3 专属 ComfyUI 插件节点
%cd /content/ComfyUI/custom_nodes
!git clone https://github.com/nkxx188/ComfyUI-MiniMaxH3-Easy.git
!git clone https://github.com/HM-RunningHub/ComfyUI_RH_MinMaxH3.git
!git clone https://github.com/ModelTC/Minimax-H3-Turbo.git

import os
if os.path.exists('/content/ComfyUI/custom_nodes/ComfyUI-MiniMaxH3-Easy/requirements.txt'):
    !pip install -q -r /content/ComfyUI/custom_nodes/ComfyUI-MiniMaxH3-Easy/requirements.txt

In [ ]:
# 6. 软链接或下载 Google Drive 中的模型权重
import os

gdrive_base = "/content/drive/MyDrive/models"
gdrive_model_path = os.path.join(gdrive_base, "minimax_h3")
comfy_model_path = "/content/ComfyUI/models/diffusion_models/minimax_h3"

!mkdir -p /content/ComfyUI/models/diffusion_models

if os.path.exists(gdrive_model_path) and os.listdir(gdrive_model_path):
    !ln -s "{gdrive_model_path}" "{comfy_model_path}"
    print(f"✅ 已成功挂载 Google Drive 中的 MiniMax-H3 模型权重至: {comfy_model_path}")
else:
    print("⚠️ Google Drive 暂无模型权重，正在创建云端硬盘目录并直接下载至 Drive 持久化保存（下次启动秒读免下载）...")
    os.makedirs(gdrive_model_path, exist_ok=True)
    !pip install -q huggingface_hub
    !huggingface-cli download MiniMaxAI/MiniMax-H3 --local-dir "{gdrive_model_path}"
    !ln -s "{gdrive_model_path}" "{comfy_model_path}"
    print("✅ MiniMax-H3 模型已保存至 Google Drive 并挂载完毕！")

In [ ]:
# 7. 安装 Cloudflare 隧道并启动 ComfyUI（输出公网可访问的画布链接）
%cd /content/ComfyUI
!wget -q -c -nc https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
!dpkg -i cloudflared-linux-amd64.deb

import subprocess
import threading
import time
import socket

def start_comfy():
    !python main.py --dont-upcast-attention --preview-method auto --use-sage-attention --listen 0.0.0.0 --port 8188

def start_tunnel():
    time.sleep(5)
    print("\n" + "="*60)
    print("🔗 正在生成公网安全穿透链接，请稍候...")
    print("="*60 + "\n")
    !cloudflared tunnel --url http://127.0.0.1:8188

threading.Thread(target=start_comfy, daemon=True).start()
time.sleep(3)
start_tunnel()